<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/huan%2Fchunking-test/chunking_strategies_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q huggingface_hub langchain_text_splitters langchain_core langchain-experimental langchain_huggingface torch tiktoken datasets

In [ ]:
from huggingface_hub import login


hf_token = ""

# Colab Secrets (🔑 in the sidebar)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass


login(token=hf_token)

In [ ]:
from huggingface_hub import whoami
print(whoami())

In [17]:
import pandas as pd
from datasets import load_dataset

repo_id = "choisongjun/Tropical_Diseases_Repo_1.3"

dataset_dict = load_dataset(repo_id, token=hf_token)
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"train cases: {len(train_df)} | test cases: {len(test_df)} | diseases: {len(DISEASE_LABELS)}")
train_df.head()

train cases: 83 | test cases: 12 | diseases: 8


,case_id,case_text,disease,images
0,PMC10870101_01,A 51-year-old female patient presented to our ...,dengue,[{'bytes': b'RIFF\xa4\x0e\x00\x00WEBPVP8 \x98\...
1,PMC6417330_01,A 24-year-old female underwent transcranial su...,dengue,[{'bytes': b'RIFFBM\x00\x00WEBPVP8 6M\x00\x00\...
2,PMC10876100_02,"A 43-year-old man, a resident of Uttar Pradesh...",dengue,[{'bytes': b'RIFF`7\x00\x00WEBPVP8 T7\x00\x00\...
3,PMC10932498_01,A 17 year old male patient with no previous co...,dengue,[{'bytes': b'RIFF\xe85\x00\x00WEBPVP8 \xdc5\x0...
4,PMC10062082_01,A 28-year-old male presented with complaint of...,dengue,[{'bytes': b'RIFF\xe6\x1b\x00\x00WEBPVP8 \xda\...


In [18]:
sample_text = train_df.loc[train_df["case_text"].str.len().idxmax(), "case_text"]

print(sample_text)

A 48-year-old Portuguese man, Caucasian, professional diver, working and living in Angola, with no significant medical history apart from several malaria episodes.
Four days after completing a 3-day prophylactic medication scheme for paludisme (sulfadoxine 500 mg/pyrimethamine 25 mg - S-P), he presented a flu-like syndrome, aggravated with fever and a pruritic facial rash, that extended to the trunk [Figure 1 resumes the progression and presentation of disease]. He was started on amoxicillin plus clavulanic acid, metronidazole, dexamethasone, loratadine, and ibuprofen. In <24 h, developed a severe mucosal damage with odynophagia and dysphagia, extensive bullous lesions on the face and trunk and a fever of 40 C. He was admitted to a medical clinic in Luanda, stopped all the previous medication, initiated fluid therapy and the day after decided to fly to Lisbon and directly to our hospital.
After evaluation by a multidisciplinary team (internal medicine and plastic surgery), a Lyell's sy

Structured Chunking

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import re

def perform_semantic_chunking(document, chunk_size=500, chunk_overlap=100):
    # Create the text splitter with semantic separators
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", r"(?<=[.!?])\s+", " ", ""],     # Look behind regular expression.
        is_separator_regex=True,                                  # Treat separators as regular expressions.
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    # Split the text into semantic chunks
    semantic_chunks = text_splitter.split_text(document)
    print(f"Document split into {len(semantic_chunks)} semantic chunks")

    # Determine section titles for enhanced metadata
    section_patterns = [
        r'^#+\s+(.+)$',      # Markdown headers
        r'^[A-Z\s]+:$'       # ALL CAPS section titles
    ]

    # Convert to Document objects with enhanced metadata
    documents = []
    current_section = "Introduction"

    for i, chunk in enumerate(semantic_chunks):
        # Try to identify section title from chunk
        chunk_lines = chunk.split('\n')
        for line in chunk_lines:
            for pattern in section_patterns:
                match = re.match(pattern, line, re.MULTILINE)
                if match:
                    current_section = match.group(0)
                    break

        # Calculate semantic density (ratio of non-stopwords to total words)
        words = re.findall(r'\b\w+\b', chunk.lower())
        stopwords = ['the', 'and', 'is', 'of', 'to', 'a', 'in', 'that', 'it', 'with', 'as', 'for']
        content_words = [w for w in words if w not in stopwords]
        semantic_density = len(content_words) / max(1, len(words))

        doc = Document(
            page_content=chunk,
            metadata={
                "chunk_id": i,
                "total_chunks": len(semantic_chunks),
                "chunk_size": len(chunk),
                "chunk_type": "semantic",
                "section": current_section,
                "semantic_density": round(semantic_density, 2)
            }
        )
        documents.append(doc)

    return documents



# Create the dummy document
document = sample_text

# Process with semantic chunking
chunked_docs = perform_semantic_chunking(document)

# Display results
print("\n----- CHUNKING RESULTS -----")
print(f"Total semantic chunks: {len(chunked_docs)}")

# Print an example chunk
print("\n----- EXAMPLE STRUCTURED CHUNK -----")
example_chunk = chunked_docs[0]
print("-" * 40)
print(example_chunk.page_content)
print("-" * 40)
print(f"Metadata: {example_chunk.metadata}")

# Optional: Calculate section distribution for analysis
section_counts = {}
for doc in chunked_docs:
    section = doc.metadata["section"]
    section_counts[section] = section_counts.get(section, 0) + 1

print("\n----- SECTION DISTRIBUTION -----")
for section, count in section_counts.items():
    print(f"{section}: {count} chunks")

Document split into 57 semantic chunks

----- CHUNKING RESULTS -----
Total semantic chunks: 57

----- EXAMPLE STRUCTURED CHUNK -----
----------------------------------------
A 48-year-old Portuguese man, Caucasian, professional diver, working and living in Angola, with no significant medical history apart from several malaria episodes.
----------------------------------------
Metadata: {'chunk_id': 0, 'total_chunks': 57, 'chunk_size': 163, 'chunk_type': 'semantic', 'section': 'Introduction', 'semantic_density': 0.83}

----- SECTION DISTRIBUTION -----
Introduction: 57 chunks


In [24]:
import torch
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", # Choose embedding model you like on Huggingface
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
)


splitter = SemanticChunker(embedding_model)

chunks = splitter.split_text(sample_text)

print("Semantic Chunks:", len(chunks))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic Chunks: 9
